In [14]:
import numpy as np
import pandas as pd
import os

In [2]:
df = pd.DataFrame()
dir = '../scraper/scraped_data/'
for file in os.listdir(dir):
    if file.endswith('.csv'):
        temp = pd.read_csv(dir + file, sep=';', decimal=',')
        df = pd.concat([df, temp], ignore_index=True)
df

,Fecha,Texto
0,2022-01-04T14:06:48.000Z,EMERGENCIA: 11:06 10-8 (LLAMADO NO CLASIFICADO...
1,2022-01-04T16:04:42.000Z,"EMERGENCIA: 13:04 10-4 (RESCATE VEHICULAR), AV..."
2,2022-01-04T17:40:50.000Z,EMERGENCIA: 14:40 10-2 (LLAMADO DE PASTIZALES)...
3,2022-01-04T17:43:44.000Z,"14:43, SALE B-11 A 10-2 (LLAMADO DE PASTIZALE..."
4,2022-01-04T17:45:44.000Z,"14:45, SALE F-7 A 10-2 (LLAMADO DE PASTIZALES..."
...,...,...
8534,2026-04-16T17:39:45.000Z,10-4-1 AVENIDA ARGENTINA / JUAN DE LA CRUZ TAP...
8535,2026-04-16T17:46:30.000Z,SALE RB-6 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...
8536,2026-04-16T17:49:42.000Z,SALE RX-5 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...
8537,2026-04-16T23:19:32.000Z,10-3-4 PORTALES / HUALPEN RB-3 https://t.co/XL...


In [24]:
import pandas as pd
df = pd.read_csv('compiled_scraped_data.csv', sep=';', decimal=',')
df['FECHA_DIA'] = df['Fecha'].astype(str).str[:10]
df['FECHA_HORA'] = df['Fecha'].astype(str).str[11:19]
patron = r"\b(10-\d+(?:-\d+)?)\b"
df['CODIGO_EMERGENCIA'] = df['Texto'].str.extract(patron)


mask_pastizal = df['Texto'].str.contains(r'PASTIZAL|FORESTAL', case=False, na=False)
mask_incendio = df['Texto'].str.contains(r'INCENDIO', case=False, na=False)

df['CODIGO_EMERGENCIA'] = np.where(
    df['CODIGO_EMERGENCIA'].notna(), df['CODIGO_EMERGENCIA'],
    np.where(mask_pastizal, '10-2-3',
    np.where(mask_incendio, '10-0-6', '0-0-0'))
)

patron = r"\b[A-Z]+-?\d+\b"  # Para códigos tipo "B5", "B-5", "RX5", "RX-5"
df['CODIGO_UNIDAD'] = df["Texto"].str.findall(patron).str.join(", ")
df

,Fecha,Texto,FECHA_DIA,FECHA_HORA,CODIGO_EMERGENCIA,CODIGO_UNIDAD
0,2022-01-04T14:06:48.000Z,EMERGENCIA: 11:06 10-8 (LLAMADO NO CLASIFICADO...,2022-01-04,14:06:48,10-8,B-1
1,2022-01-04T16:04:42.000Z,"EMERGENCIA: 13:04 10-4 (RESCATE VEHICULAR), AV...",2022-01-04,16:04:42,10-4,"RB-5, B-9"
2,2022-01-04T17:40:50.000Z,EMERGENCIA: 14:40 10-2 (LLAMADO DE PASTIZALES)...,2022-01-04,17:40:50,10-2,B-4
3,2022-01-04T17:43:44.000Z,"14:43, SALE B-11 A 10-2 (LLAMADO DE PASTIZALE...",2022-01-04,17:43:44,10-2,B-11
4,2022-01-04T17:45:44.000Z,"14:45, SALE F-7 A 10-2 (LLAMADO DE PASTIZALES...",2022-01-04,17:45:44,10-2,F-7
...,...,...,...,...,...,...
8534,2026-04-16T17:39:45.000Z,10-4-1 AVENIDA ARGENTINA / JUAN DE LA CRUZ TAP...,2026-04-16,17:39:45,10-4-1,"RB-3, BQ-7"
8535,2026-04-16T17:46:30.000Z,SALE RB-6 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...,2026-04-16,17:46:30,10-4-1,RB-6
8536,2026-04-16T17:49:42.000Z,SALE RX-5 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...,2026-04-16,17:49:42,10-4-1,RX-5
8537,2026-04-16T23:19:32.000Z,10-3-4 PORTALES / HUALPEN RB-3 https://t.co/XL...,2026-04-16,23:19:32,10-3-4,RB-3


In [27]:
codigos = pd.read_excel('Clave_CBT.xlsx')
df1 = pd.merge(df, codigos, how='left', on='CODIGO_EMERGENCIA')
df1 = df1.drop(columns='DESCRIPCION')
df1

,Fecha,Texto,FECHA_DIA,FECHA_HORA,CODIGO_EMERGENCIA,CODIGO_UNIDAD,CATEGORIA_EMERGENCIA
0,2022-01-04T14:06:48.000Z,EMERGENCIA: 11:06 10-8 (LLAMADO NO CLASIFICADO...,2022-01-04,14:06:48,10-8,B-1,LLAMADOS SIN SUBCLASIFICACIÓN
1,2022-01-04T16:04:42.000Z,"EMERGENCIA: 13:04 10-4 (RESCATE VEHICULAR), AV...",2022-01-04,16:04:42,10-4,"RB-5, B-9",RESCATE VEHICULAR
2,2022-01-04T17:40:50.000Z,EMERGENCIA: 14:40 10-2 (LLAMADO DE PASTIZALES)...,2022-01-04,17:40:50,10-2,B-4,INCENDIO PASTIZAL O FORESTAL
3,2022-01-04T17:43:44.000Z,"14:43, SALE B-11 A 10-2 (LLAMADO DE PASTIZALE...",2022-01-04,17:43:44,10-2,B-11,INCENDIO PASTIZAL O FORESTAL
4,2022-01-04T17:45:44.000Z,"14:45, SALE F-7 A 10-2 (LLAMADO DE PASTIZALES...",2022-01-04,17:45:44,10-2,F-7,INCENDIO PASTIZAL O FORESTAL
...,...,...,...,...,...,...,...
10489,2026-04-16T17:39:45.000Z,10-4-1 AVENIDA ARGENTINA / JUAN DE LA CRUZ TAP...,2026-04-16,17:39:45,10-4-1,"RB-3, BQ-7",RESCATE VEHICULAR
10490,2026-04-16T17:46:30.000Z,SALE RB-6 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...,2026-04-16,17:46:30,10-4-1,RB-6,RESCATE VEHICULAR
10491,2026-04-16T17:49:42.000Z,SALE RX-5 A 10-4-1 AVENIDA ARGENTINA / JUAN DE...,2026-04-16,17:49:42,10-4-1,RX-5,RESCATE VEHICULAR
10492,2026-04-16T23:19:32.000Z,10-3-4 PORTALES / HUALPEN RB-3 https://t.co/XL...,2026-04-16,23:19:32,10-3-4,RB-3,RESCATE DE PERSONAS


In [29]:
df.to_csv('tweets_procesados.csv', index=False, sep=';', decimal=',')